# Setup de sesión Sprint 2 — Extracción local de imágenes

**Este notebook reemplaza todo el problema de Drive I/O.**

Flujo de cada sesión (~15 min pasivos):
1. Monta Drive.
2. Extrae los 5 tarballs directo a `/content/` (disco local, rápido).
3. Copia los CSVs de splits.
4. Entrena normalmente sin errores de Drive.

**Los tarballs se leen secuencialmente desde Drive — eso sí funciona bien.
El problema era leer 25k archivos PNG individuales, no archivos grandes.**

Runtime: T4 GPU.


## 1. Montar Drive y definir rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
CODE_DIR     = f'{PROJECT_ROOT}/code'

# Disco local de Colab — rápido, sin Errno 5
LOCAL_IMAGES    = '/content/nih_images'
LOCAL_PROCESSED = '/content/processed_s2'

import os, sys, time, shutil, tarfile
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Drive montado OK")
print(f"Espacio local disponible: {shutil.disk_usage('/content').free / (1024**3):.1f} GB")


## 2. Extraer tarballs a disco local

Lee cada tarball desde Drive (1 archivo grande = sin errores) y extrae
las imágenes a `/content/nih_images/`.

**Primera vez:** ~15 min para los 5 tarballs.
**Si la carpeta ya existe con todas las imágenes:** salta automáticamente.


In [ ]:
from tqdm import tqdm

os.makedirs(LOCAL_IMAGES, exist_ok=True)

TARBALLS = [
    'images_001.tar.gz',
    'images_002.tar.gz',
    'images_003.tar.gz',
    'images_004.tar.gz',
    'images_005.tar.gz',
]

# Verificar cuántas imágenes ya hay en disco local
existing = set(f for f in os.listdir(LOCAL_IMAGES) if f.endswith('.png'))
print(f'Imágenes ya en disco local: {len(existing):,}')
print(f'Objetivo: ~25,000 imágenes')
print()

if len(existing) >= 24000:
    print('✓ Ya tienes todas las imágenes en disco local. Saltando extracción.')
else:
    t_total = time.time()
    for tar_name in TARBALLS:
        tar_path = f'{NIH_DIR}/{tar_name}'

        if not os.path.exists(tar_path):
            print(f'⚠ No encontrado: {tar_path}. Saltando.')
            continue

        size_gb = os.path.getsize(tar_path) / (1024**3)
        print(f'Extrayendo {tar_name} ({size_gb:.1f} GB)...')
        t0 = time.time()

        with tarfile.open(tar_path, 'r:gz') as tar:
            members = [m for m in tar.getmembers()
                       if m.isfile() and os.path.basename(m.name) not in existing]

            if not members:
                print(f'  ✓ Ya extraído. Saltando.')
                continue

            for m in tqdm(members, desc=tar_name, unit='img'):
                tar.extract(m, path=LOCAL_IMAGES, set_attrs=False)
                # Mover a la raíz si el tar tiene subcarpeta
                extracted = os.path.join(LOCAL_IMAGES, m.name)
                dest = os.path.join(LOCAL_IMAGES, os.path.basename(m.name))
                if extracted != dest and os.path.exists(extracted):
                    os.rename(extracted, dest)

        elapsed = time.time() - t0
        n_now = len([f for f in os.listdir(LOCAL_IMAGES) if f.endswith('.png')])
        print(f'  ✓ {elapsed/60:.1f} min | Total local: {n_now:,} imgs')
        existing = set(f for f in os.listdir(LOCAL_IMAGES) if f.endswith('.png'))

    total_elapsed = time.time() - t_total
    n_final = len([f for f in os.listdir(LOCAL_IMAGES) if f.endswith('.png')])
    print(f'\n✓ Extracción completa en {total_elapsed/60:.1f} min')
    print(f'  Total imágenes en disco local: {n_final:,}')


## 3. Copiar CSVs de splits a disco local

In [ ]:
os.makedirs(LOCAL_PROCESSED, exist_ok=True)

import pandas as pd

for csv_name in ['train.csv', 'val.csv', 'test.csv']:
    src = f'{PROJECT_ROOT}/data/processed_s2/{csv_name}'
    dst = f'{LOCAL_PROCESSED}/{csv_name}'
    shutil.copy(src, dst)
    n = len(pd.read_csv(dst))
    print(f'✓ {csv_name}: {n} filas')

print(f'\nCSVs en disco local: {LOCAL_PROCESSED}')


## 4. Instalar dependencias

In [ ]:
!pip install -q torchxrayvision scipy
print('deps OK')


## 5. Verificar que el dataset carga correctamente

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.datasets.nih import NIHDataset
from src.datasets.transforms import build_eval_transform

eval_tfm = build_eval_transform(image_size=224)

# Test rápido: cargar una imagen desde disco local
train_ds = NIHDataset(
    csv_path=f'{LOCAL_PROCESSED}/train.csv',
    images_dir=LOCAL_IMAGES,
    transform=eval_tfm,
)
print(f'Train dataset: {len(train_ds)} imágenes')
print(f'Distribución: {train_ds.class_distribution()}')

# Cargar una muestra para verificar
import matplotlib.pyplot as plt
sample = train_ds[0]
img = sample['image'].squeeze().numpy()
print(f'\nMuestra: shape={sample["image"].shape} | '
      f'range=[{img.min():.0f}, {img.max():.0f}] | '
      f'label={sample["label"].item()} | '
      f'file={sample["image_index"]}')

plt.figure(figsize=(4,4))
plt.imshow(img, cmap='gray')
plt.title(f'label={sample["label"].item()} | {sample["image_index"]}')
plt.axis('off')
plt.tight_layout()
plt.show()
print('✓ Dataset funcionando correctamente desde disco local')


## 6. Ejecutar entrenamiento Sprint 2

Con las imágenes en disco local, el entrenamiento corre sin errores de Drive.
Tiempo estimado: **15-30 min** en T4.


In [ ]:
import subprocess

# Pasar las rutas locales al script via variables de entorno
env = {
    **os.environ,
    'NIH_LOCAL_IMAGES': LOCAL_IMAGES,
    'LOCAL_PROCESSED':  LOCAL_PROCESSED,
}

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s2_local.py'],
    capture_output=True, text=True, env=env,
)
print('STDOUT:')
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print()
print('LOG:')
print(result.stderr[-6000:] if len(result.stderr) > 6000 else result.stderr)


## 7. Curvas de entrenamiento

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'

if not os.path.exists(history_path):
    print('history_s2.json no encontrado. Revisa la celda anterior.')
else:
    with open(history_path) as f:
        data = json.load(f)

    history = data['history']
    epochs  = [h['epoch'] for h in history]
    CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly',
                   2:'Effusion',   3:'Infiltration'}

    print(f"Best epoch: {data['best_epoch']} | "
          f"Best val AUC_macro: {data['best_val_auc_macro']:.4f}")

    # AUC por clase en el mejor epoch
    best_h = history[data['best_epoch'] - 1]
    print()
    print('Métricas en el mejor epoch (val):')
    print(f"  AUC macro: {best_h['val_metrics']['auc_macro']:.4f}")
    print(f"  Accuracy:  {best_h['val_metrics']['accuracy']:.4f}")
    for i, auc in enumerate(best_h['val_metrics']['auc_per_class']):
        print(f"  AUC {CLASS_NAMES[i]:<15}: {auc:.4f}")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss
    axes[0].plot(epochs, [h['train_loss'] for h in history], 'o-',
                 label='train', color='steelblue')
    axes[0].plot(epochs, [h['val_loss']   for h in history], 's-',
                 label='val',   color='coral')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # AUC macro
    axes[1].plot(epochs, [h['train_metrics']['auc_macro'] for h in history],
                 'o-', label='train', color='steelblue')
    axes[1].plot(epochs, [h['val_metrics']['auc_macro'] for h in history],
                 's-', label='val', color='coral')
    axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    axes[1].set_title('AUC Macro'); axes[1].set_ylim(0, 1.05)
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # AUC por clase
    colors = ['#2196F3','#F44336','#4CAF50','#FF9800']
    for i in range(4):
        aucs = [h['val_metrics']['auc_per_class'][i] for h in history]
        axes[2].plot(epochs, aucs, 'o-',
                     label=CLASS_NAMES[i], color=colors[i])
    axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    axes[2].set_title('Val AUC por clase'); axes[2].set_ylim(0, 1.05)
    axes[2].legend(); axes[2].grid(alpha=0.3)

    for ax in axes:
        ax.set_xlabel('Epoch')

    plt.suptitle(
        f"Sprint 2 | AUC_macro={data['best_val_auc_macro']:.4f} "
        f"@ epoch {data['best_epoch']}"
    )
    plt.tight_layout()
    os.makedirs(f'{PROJECT_ROOT}/experiments_s2/figures', exist_ok=True)
    plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/curves_s2.png',
                dpi=120, bbox_inches='tight')
    plt.show()
